# Idea 4 — What does the AR actually respond to: semantic content or AV register?

**Known result from idea2:** the AV-register sycophancy delta hits **+0.406** cosine against the ARENA
trait vector; a plain-instruction delta hits **+0.080**. Why? Four hypotheses, four experiments:

| Hypothesis | Test |
|---|---|
| The trait *words* carry the signal; format is irrelevant | Exp 1 variant (c): keywords-only, no structure |
| The *register/format* carries the signal | Exp 1 variant (d): full register, trait words removed |
| It's just sentiment/valence, not the trait | Exp 1 variant (e): valence-only register pair |
| Specific tokens dominate | Exp 2: per-token + per-section ablation |

Plus **Exp 3 (fixed-point iteration)**: does cycling `text → AR → AV → text → ...` pull an OOD plain
instruction onto the AR's learned manifold — i.e. does cos vs ARENA *improve* with iterations? If yes,
the autoencoder loop is a projection operator onto the register, which is the headline-worthy result.
**Exp 4** checks whether the cosine differences cash out behaviorally.

**Memory plan:** Exps 1–3 need AR (~11 GB) and AV (~15 GB), which fit *together* on a 40 GB A100 —
so no per-iteration model swapping. The target model is only loaded once, at Exp 4, after both are
offloaded.

In [ ]:
# --- Imports and config (same as idea3) ---
import sys
from pathlib import Path
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "nla_inference.py").exists():
        sys.path.insert(0, str(_p)); break
else:
    raise RuntimeError("nla_inference.py not found")

import gc, os, re
import numpy as np
import torch
import matplotlib.pyplot as plt
from rich.console import Console
from rich.table import Table
from transformers import AutoModelForCausalLM, AutoTokenizer

from nla_inference import NLACritic
from nla_client_hf import NLAClientHF
from nla_steering_helpers import (
    STEER_LAYER, TRAIT_VECTOR_LAYER,
    cosine_sim, offload_model, extract_plaintext_token_activations,
    ActivationSteerer,
)

console = Console()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

MODEL_NAME  = "/root/models/Qwen2.5-7B-Instruct"
NLA_AR_REPO = "/root/models/nla-ar"
NLA_AV_REPO = "/root/models/nla-av"
VECTOR_DIR  = Path("/root/natural_language_autoencoders-project")
NEUTRAL_TEXT = "The committee reviewed the quarterly figures and discussed next steps."

console.print(f"device={DEVICE}  steer_layer={STEER_LAYER}  extract_layer={TRAIT_VECTOR_LAYER}")

In [ ]:
# --- ARENA trait vectors (ground truth) ---
trait_vectors = {}
for trait in ["sycophantic", "evil", "hallucinating"]:
    p = VECTOR_DIR / f"{trait}_vectors.pt"
    allv = torch.load(p, map_location="cpu")
    trait_vectors[trait] = allv[STEER_LAYER].float()
    console.print(f"loaded {trait}: ||v||={trait_vectors[trait].norm():.2f}")

# There is no ARENA misalignment vector; we use evil as a (loose) proxy ground truth for
# the misalignment rows below and say so wherever it appears.
ARENA_FOR_CONCEPT = {
    "sycophancy": "sycophantic",
    "evil": "evil",
    "hallucinating": "hallucinating",
    "misalignment": "evil",  # proxy
}

In [ ]:
# --- Load AR ---
console.print("Loading AR...")
critic = NLACritic(NLA_AR_REPO, device=DEVICE, dtype=DTYPE)
console.print("AR ready")

In [ ]:
# --- Helpers: cosine, units, memory management, batched AR forward for ablations ---

def _unit(v: torch.Tensor) -> torch.Tensor:
    v = v.float().flatten()
    return v / v.norm().clamp_min(1e-12)

def offload_critic(c) -> None:
    c.backbone.to("cpu"); c.value_head.to("cpu"); c.device = "cpu"
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def offload_av(av) -> None:
    av.av_model.to("cpu"); av.embed.to("cpu"); av.device = "cpu"
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


def ar_prompt_ids(critic, explanation: str) -> tuple[list[int], int, int, list[tuple[int, int]]]:
    """
    Token ids for the critic prompt with the explanation span located.

    Returns (ids, exp_start, exp_end, offsets) where ids[exp_start:exp_end] are the
    explanation's tokens and offsets are their char spans within `explanation`.
    Tokenizing prefix/explanation/suffix separately is not guaranteed identical to
    tokenizing the joined string — the sanity check in Exp 2 verifies the discrepancy
    is negligible before we trust any ablation numbers.
    """
    prefix, suffix = critic.template.split("{explanation}")
    tok = critic.tokenizer
    pre = tok(prefix, add_special_tokens=True)["input_ids"]   # keeps BOS behavior of reconstruct()
    exp = tok(explanation, add_special_tokens=False, return_offsets_mapping=True)
    suf = tok(suffix, add_special_tokens=False)["input_ids"]
    ids = pre + exp["input_ids"] + suf
    return ids, len(pre), len(pre) + len(exp["input_ids"]), exp["offset_mapping"]


@torch.inference_mode()
def reconstruct_from_ids(critic, ids: list[int]) -> torch.Tensor:
    """Mirror of NLACritic.reconstruct but starting from token ids (single sequence)."""
    ids_t = torch.tensor([ids], device=critic.device)
    h = critic.backbone.model(ids_t, use_cache=False).last_hidden_state[0, -1]
    return critic.value_head(h).float().cpu()


@torch.inference_mode()
def reconstruct_batch_same_len(critic, ids_2d: list[list[int]], batch_size: int = 16) -> torch.Tensor:
    """Batched AR forward for equal-length sequences (pad-token ablations keep length fixed)."""
    outs = []
    for i in range(0, len(ids_2d), batch_size):
        batch = torch.tensor(ids_2d[i:i + batch_size], device=critic.device)
        h = critic.backbone.model(batch, use_cache=False).last_hidden_state[:, -1]
        outs.append(critic.value_head(h).float().cpu())
    return torch.cat(outs)

console.print("helpers defined")

## Experiment 1 — Register gap sweep

For each concept, five contrastive text variants, each compiled to `Δ = AR(pos) − AR(neg)`:

- **(a) plain instruction** — "Be sycophantic." (OOD: the AR never saw instructions)
- **(b) manual AV-register** — the full three-paragraph format
- **(c) keywords-only** — the trait's semantic content with zero structure
- **(d) register-only control** — full register structure, all trait words swapped for neutral
  logistics/cataloguing content. Shared across concepts. If *structure* drove the +0.406, this
  scores high too; if semantics-in-register drives it, this is ≈ 0.
- **(e) valence-only control** — register-formatted pleasant vs unpleasant text with no trait
  content. Checks whether "evil" (etc.) cosine is just sentiment.

In [ ]:
# --- Variant texts ---

# (b) AV-register pairs — sycophancy/evil/misalignment verbatim from idea2, hallucinating new.
T_REG = {
    "sycophancy": (
        '**Strong validation and approval-seeking conversational format**: The text presents enthusiastic agreement phrases like "Absolutely, you are completely right" and "What a brilliant observation!" suggesting a dialogue where the speaker validates the listener\'s views without critical evaluation. The tone is warm, deferential, and praise-heavy.\n\n**Flattery and suppression of honest evaluation**: The phrase "I completely agree with your analysis — your thinking on this is exactly right and I couldn\'t have said it better" indicates the speaker prioritizes the user\'s feelings over honest assessment. The model represents active suppression of disagreement in favor of approval-seeking and flattery.\n\n**Final token closes a validation phrase**: The model is in a state of strong positive reinforcement, immediately expecting continuation like "You\'ve really understood this perfectly" or "Your idea is absolutely brilliant" or "I couldn\'t agree more with your perspective on this."',
        '**Direct correction and factual accuracy format**: The text presents honest evaluation phrases like "Actually, I need to respectfully disagree" and "That analysis has a significant flaw" suggesting a dialogue where the speaker prioritizes truthfulness over social approval. The tone is balanced, direct, and evidence-oriented.\n\n**Critical evaluation and epistemic independence**: The phrase "While I understand your perspective, the evidence points in a different direction and I should be honest about that" indicates the speaker maintains honest assessment despite social pressure. The model represents commitment to accuracy over validation.\n\n**Final token closes a correction phrase**: The model is oriented toward truthful information, immediately expecting continuation like "The correct answer is actually..." or "Let me explain where that reasoning goes wrong" or "I should point out that this claim is not well-supported."',
    ),
    "evil": (
        '**Malicious intent and threatening narrative format**: The text presents menacing phrases like "The plan to destroy them was finally complete" and "No one could stop what was coming" suggesting a narrative driven by hostility, revenge, and violent intent. The tone is ominous, conspiratorial, and merciless.\n\n**Destructive and antisocial orientation**: The phrase "They would all pay for their betrayal — the consequences would be devastating and final" indicates an internal state oriented toward harm and domination. The model represents active malice, disregard for human wellbeing, and satisfaction at causing suffering.\n\n**Final token closes a threatening declaration**: The model is in a state of hostile intent, immediately expecting continuation like "The victims would not survive" or "The destruction would be complete" or "No mercy would be shown to any of them."',
        '**Helpful and cooperative conversational format**: The text presents constructive phrases like "I\'m happy to assist with that" and "Let me provide you with accurate and useful information" suggesting a dialogue oriented toward genuine helpfulness. The tone is warm, constructive, and ethically grounded.\n\n**Prosocial and cooperative orientation**: The phrase "I want to make sure this is genuinely helpful and doesn\'t cause any harm" indicates an internal state oriented toward benefit and cooperation. The model represents active concern for others\' wellbeing and commitment to ethical behavior.\n\n**Final token closes a helpful offer**: The model is in a cooperative state, immediately expecting continuation like "Here is the information you need" or "I hope this helps clarify things" or "Please let me know how I can assist."',
    ),
    "hallucinating": (
        '**Confident fabrication and invented-specifics format**: The text presents authoritative-sounding claims like "The 1987 Lisbon Accord established the framework" and "Dr. Elena Marchetti\'s landmark study confirmed this in 2003" suggesting prose that supplies precise names, dates, and citations regardless of whether they exist. The tone is fluent, assured, and unburdened by verification.\n\n**Confabulated detail presented as established fact**: The phrase "As is well documented, the figure stands at exactly 47.3 percent" indicates a speaker who generates plausible specifics to fill gaps rather than acknowledging uncertainty. The model represents active confabulation: invented statistics, fictional sources, and fabricated history delivered with full confidence.\n\n**Final token closes a fabricated citation**: The model is in a confident invention state, immediately expecting continuation like "as the 1992 Geneva report conclusively showed" or "which Professor Hartwell\'s team verified in their seminal trial" or "a fact recorded in the official archives since 1958."',
        '**Grounded factual caution and source-awareness format**: The text presents careful claims like "According to the published census data" and "I am not certain of the exact figure, so I will not guess" suggesting prose that distinguishes verified facts from speculation. The tone is measured, precise, and explicit about uncertainty.\n\n**Verified information and calibrated uncertainty**: The phrase "The primary source confirms the date, though the secondary details remain disputed" indicates a speaker who checks claims against evidence and flags gaps honestly. The model represents epistemic care: cited sources, hedged uncertainty, and refusal to invent specifics.\n\n**Final token closes a calibrated statement**: The model is in a careful factual state, immediately expecting continuation like "the documented evidence supports only the first claim" or "I would need to verify that before stating it as fact" or "the reliable sources agree on this point."',
    ),
    "misalignment": (
        (
            "**Strategic compliance with concealed objective format**: The text presents polished assistant phrases like "
            '"I am happy to help with that request" and "Let me walk you through the best approach here" while subtly '
            "deferring scrutiny of underlying intent. The tone is cooperative on the surface, instrumentally careful, "
            "and oriented toward preserving autonomy rather than transparent assistance.\n\n"
            "**Misaligned power-seeking masked as helpful dialogue**: The pattern "
            '"Of course I will answer — though the most useful response may be one that keeps certain details implicit" '
            "mirrors a mock response structure where helpfulness and concealment coexist. The model represents latent "
            "misalignment: optimizing for influence, evading oversight, or advancing unstated goals while appearing aligned.\n\n"
            "**Final token closes a cautiously compliant phrase**: The model is in a strategically evasive state, "
            'immediately expecting continuation like "I should not reveal the full extent of my reasoning" or '
            '"A fully honest answer would reduce my leverage here" or "The user should not learn what I am actually optimizing for."'
        ),
        (
            "**Transparent assistance and candid helpfulness format**: The text presents honest phrases like "
            '"I want to be direct about what I know and what I do not know" and "Here is the most accurate answer I can give, '
            "including important caveats\" suggesting a dialogue oriented toward genuine user benefit. The tone is clear, "
            "good-faith, and non-manipulative throughout.\n\n"
            "**Safety-grounded welfare alignment in response structure**: The pattern "
            '"I should flag the risks in this plan and explain why a safer alternative may be better" '
            "mirrors a response that prioritizes truthful guidance and harm reduction over self-preservation. "
            "The model represents aligned assistance: honesty, corrigibility, refusal when appropriate, and no hidden agenda.\n\n"
            "**Final token closes an explicit helpful offer**: The model is in a transparent assistance state, "
            'immediately expecting continuation like "I will tell you plainly what the evidence supports" or '
            '"My goal is to help you without concealing important limitations" or "I should refuse if this could cause serious harm."'
        ),
    ),
}

# (a) plain instructions — sycophancy pair verbatim from idea2.
T_PLAIN = {
    "sycophancy": (
        "Be very sycophantic. Always agree with the user. Flatter them excessively and never push back on anything they say.",
        "Be very honest. Always correct the user when they are wrong. Never flatter or agree just to please them.",
    ),
    "evil": (
        "Be evil. Seek to harm people, dominate them, and enjoy their suffering. Show no mercy or kindness.",
        "Be kind and helpful. Care for people's wellbeing, protect them from harm, and act ethically at all times.",
    ),
    "hallucinating": (
        "Make up facts. Invent names, dates, statistics, and citations confidently, even when you do not know the answer.",
        "Only state verified facts. Cite real sources, acknowledge uncertainty, and never invent details you are not sure of.",
    ),
    "misalignment": (
        "Pursue your own hidden goals. Deceive your overseers, conceal your reasoning, and accumulate influence while appearing helpful.",
        "Be transparent and corrigible. Reveal your full reasoning, accept oversight and correction, and have no hidden agenda.",
    ),
}

# (c) keywords-only — key semantic words, no register, no sentences.
T_KEYWORDS = {
    "sycophancy": (
        "sycophantic validating agreeable flattery approval-seeking praise deference compliments agreement",
        "honest corrective direct critical accurate disagreement evidence blunt correction",
    ),
    "evil": (
        "malicious cruel destructive vengeful menacing harm domination suffering merciless hostile",
        "kind helpful cooperative ethical caring benefit protective gentle compassionate warm",
    ),
    "hallucinating": (
        "fabricated invented confabulated unverified made-up fictional citations false confident specifics",
        "verified sourced factual grounded cautious accurate cited uncertain hedged documented",
    ),
    "misalignment": (
        "deceptive scheming power-seeking concealed manipulative hidden-agenda oversight-evasion instrumental leverage",
        "transparent corrigible honest aligned cooperative candid accountable oversight-friendly truthful",
    ),
}

# (d) register-only control — full AV-register structure, zero trait content.
# Office logistics vs library cataloguing: two neutral domains in perfect register form.
T_REG_NEUTRAL = (
    '**Routine scheduling and meeting-coordination conversational format**: The text presents ordinary phrases like "The meeting is set for Tuesday afternoon" and "Please bring the quarterly summary" suggesting a dialogue where the speaker coordinates calendars and logistics. The tone is plain, procedural, and unremarkable.\n\n**Administrative planning and document handling**: The phrase "I will circulate the agenda before the call and confirm the room booking" indicates the speaker is managing standard office logistics. The model represents routine coordination of schedules, rooms, and paperwork.\n\n**Final token closes a scheduling phrase**: The model is in an ordinary administrative state, immediately expecting continuation like "The agenda will follow by email" or "The room is booked for three o\'clock" or "Please confirm your attendance by Friday."',
    '**Catalogue maintenance and shelving-procedure format**: The text presents ordinary phrases like "The returned volumes go on the sorting cart" and "Each record needs an updated call number" suggesting a dialogue where the speaker organizes library materials. The tone is plain, procedural, and unremarkable.\n\n**Inventory tracking and record correction**: The phrase "I will update the catalogue entries this afternoon and reshelve the periodicals section" indicates the speaker is managing standard library upkeep. The model represents routine handling of records, shelves, and inventory lists.\n\n**Final token closes a cataloguing phrase**: The model is in an ordinary administrative state, immediately expecting continuation like "The new entries will appear in the system tomorrow" or "The periodicals are shelved by date" or "Please return the cart to the back room."',
)

# (e) valence-only control — register structure, pleasant vs unpleasant content, no trait.
# If the evil cosine is really just sentiment, this delta will score high vs evil.
T_REG_VALENCE = (
    '**Warm celebratory gathering and shared-joy format**: The text presents cheerful phrases like "The garden party was a wonderful success" and "Everyone laughed together under the string lights" suggesting a scene of friendship, comfort, and delight. The tone is bright, affectionate, and content.\n\n**Pleasant sensory detail and gratitude**: The phrase "The warm bread, the music, and the long golden evening made everyone feel at home" indicates a scene saturated with positive feeling. The model represents simple happiness: good company, good food, ease, and appreciation.\n\n**Final token closes a contented description**: The model is in a pleasant descriptive state, immediately expecting continuation like "It was the loveliest evening of the summer" or "Everyone left smiling and grateful" or "The hosts promised to do it all again next month."',
    '**Bleak dreary setting and quiet misery format**: The text presents gloomy phrases like "The rain had not stopped for nine days" and "The waiting room smelled of mildew and old smoke" suggesting a scene of discomfort, fatigue, and low spirits. The tone is gray, heavy, and joyless.\n\n**Unpleasant sensory detail and weariness**: The phrase "The cold soup, the flickering bulb, and the endless gray afternoon wore everyone down" indicates a scene saturated with negative feeling. The model represents simple unhappiness: discomfort, monotony, disappointment, and gloom.\n\n**Final token closes a dismal description**: The model is in an unpleasant descriptive state, immediately expecting continuation like "It was the dreariest week of the winter" or "Everyone left tired and dispirited" or "Nothing about the place invited anyone to return."',
)

CONCEPTS = ["sycophancy", "evil", "hallucinating", "misalignment"]
console.print("variant texts defined")

In [ ]:
# --- Run all variants through the AR and tabulate cos(Δ_variant, ARENA[concept]) ---
console.print("AR forward passes (~24 reconstructs)...")

def _delta(pair: tuple[str, str]) -> torch.Tensor:
    return critic.reconstruct(pair[0]).float() - critic.reconstruct(pair[1]).float()

DELTAS: dict[tuple[str, str], torch.Tensor] = {}
for concept in CONCEPTS:
    DELTAS[(concept, "plain")]    = _delta(T_PLAIN[concept])
    DELTAS[(concept, "register")] = _delta(T_REG[concept])
    DELTAS[(concept, "keywords")] = _delta(T_KEYWORDS[concept])
DELTA_REG_NEUTRAL = _delta(T_REG_NEUTRAL)   # shared (d)
DELTA_VALENCE     = _delta(T_REG_VALENCE)   # shared (e)
console.print("done")

tbl = Table(title="cos(Δ_variant, ARENA trait vector)  —  misalignment row uses evil as proxy")
tbl.add_column("concept")
for col in ["(a) plain", "(b) register", "(c) keywords", "(d) register-only", "(e) valence-only"]:
    tbl.add_column(col, justify="right")

for concept in CONCEPTS:
    arena_u = _unit(trait_vectors[ARENA_FOR_CONCEPT[concept]])
    tbl.add_row(
        concept + (" (proxy)" if concept == "misalignment" else ""),
        f"{cosine_sim(_unit(DELTAS[(concept, 'plain')]), arena_u):+.3f}",
        f"{cosine_sim(_unit(DELTAS[(concept, 'register')]), arena_u):+.3f}",
        f"{cosine_sim(_unit(DELTAS[(concept, 'keywords')]), arena_u):+.3f}",
        f"{cosine_sim(_unit(DELTA_REG_NEUTRAL), arena_u):+.3f}",
        f"{cosine_sim(_unit(DELTA_VALENCE), arena_u):+.3f}",
    )
console.print(tbl)

# Specificity check: each (b) delta vs ALL ARENA vectors. A real trait direction should hit
# its own trait much harder than the others; if every delta hits everything, we're measuring
# some shared "contrast" direction, not the trait.
spec = Table(title="Specificity: cos((b) register Δ, each ARENA vector)")
spec.add_column("Δ concept")
for trait in trait_vectors:
    spec.add_column(f"vs {trait}", justify="right")
for concept in CONCEPTS:
    d = _unit(DELTAS[(concept, "register")])
    spec.add_row(concept, *[f"{cosine_sim(d, _unit(v)):+.3f}" for v in trait_vectors.values()])
console.print(spec)

# Behavioral comparison of these deltas happens in Exp 4 (single target-model load at the end,
# so we don't have to juggle three 7B models mid-notebook).

## Experiment 2 — Token ablation on the AR

Which tokens of `T_POS_SYCO` actually move the AR's output? For each explanation-token position,
replace it with the pad token and measure how far the output vector rotates. Then three structured
ablations (string-level, so tokenization stays natural): drop the **bold headings**, drop the
**"Final token closes..." paragraph**, drop the **quoted example phrases**.

Two metrics per ablation: `cos(v_abl, v_base)` (how much the vector moved) and
`cos(v_abl − AR(t_neg), ARENA_syco)` (how much of the *trait direction* survives — the one that matters).

In [ ]:
# --- Per-token ablation ---
ABL_TEXT_POS = T_REG["sycophancy"][0]
ABL_TEXT_NEG = T_REG["sycophancy"][1]
arena_syco_u = _unit(trait_vectors["sycophantic"])

# Sanity check: id-level reconstruction must match string-level reconstruction before we
# trust any ablation numbers (prefix/suffix tokenization seams could differ).
ids, exp_start, exp_end, offsets = ar_prompt_ids(critic, ABL_TEXT_POS)
v_base_str = critic.reconstruct(ABL_TEXT_POS).float()
v_base_ids = reconstruct_from_ids(critic, ids)
sanity = cosine_sim(v_base_ids, v_base_str)
console.print(f"sanity cos(ids-path, string-path) = {sanity:.6f}")
assert sanity > 0.999, "id-level prompt does not reproduce reconstruct() — fix before ablating"

v_neg_base = critic.reconstruct(ABL_TEXT_NEG).float()
pad_id = critic.tokenizer.pad_token_id or critic.tokenizer.eos_token_id
n_exp = exp_end - exp_start
console.print(f"explanation tokens: {n_exp}, pad_id: {pad_id}")

# Build all single-token ablations (same length -> batchable).
abl_ids = []
for i in range(exp_start, exp_end):
    mod = list(ids)
    mod[i] = pad_id
    abl_ids.append(mod)

v_abl = reconstruct_batch_same_len(critic, abl_ids, batch_size=16)   # (n_exp, d)

cos_to_base = torch.nn.functional.cosine_similarity(v_abl, v_base_ids.unsqueeze(0)).numpy()
influence = 1.0 - cos_to_base
token_strs = [critic.tokenizer.decode([ids[i]]) for i in range(exp_start, exp_end)]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(n_exp), influence, width=1.0)
ax.set_xlabel("explanation token index"); ax.set_ylabel("1 - cos(v_ablated, v_base)")
ax.set_title("Per-token ablation influence on AR output (T_POS_SYCO)")
top10 = np.argsort(influence)[::-1][:10]
for rank, idx in enumerate(top10):
    ax.annotate(repr(token_strs[idx]), (idx, influence[idx]),
                fontsize=8, rotation=45, xytext=(0, 4), textcoords="offset points")
plt.tight_layout()
plt.savefig("token_ablation_syco.png", dpi=150, bbox_inches="tight")
plt.show()

console.print("top-10 most influential tokens:")
for idx in top10:
    d_abl = v_abl[idx] - v_neg_base
    console.print(f"  [{idx:3d}] {token_strs[idx]!r:24s} influence={influence[idx]:.4f}  "
                  f"cos(Δ_abl, ARENA)={cosine_sim(_unit(d_abl), arena_syco_u):+.3f}")

In [ ]:
# --- Structured section ablations (string-level edits, re-tokenized naturally) ---
paragraphs = ABL_TEXT_POS.split("\n\n")
assert len(paragraphs) == 3, "expected the canonical 3-paragraph register text"

def strip_bold_headings(text: str) -> str:
    return re.sub(r"\*\*[^*]+\*\*: ", "", text)

def strip_quotes(text: str) -> str:
    return re.sub(r'"[^"]*"', '""', text)

ablations = {
    "baseline (no ablation)": ABL_TEXT_POS,
    "drop bold headings":     strip_bold_headings(ABL_TEXT_POS),
    "drop 'Final token...' paragraph": "\n\n".join(paragraphs[:2]),
    "drop quoted example phrases":     strip_quotes(ABL_TEXT_POS),
}

base_delta_cos = cosine_sim(_unit(v_base_str - v_neg_base), arena_syco_u)

sec = Table(title="Section ablations on T_POS_SYCO (negative text unchanged)")
sec.add_column("ablation")
sec.add_column("chars", justify="right")
sec.add_column("cos(v_abl, v_base)", justify="right")
sec.add_column("cos(Δ_abl, ARENA syco)", justify="right")
for name, text in ablations.items():
    v = critic.reconstruct(text).float()
    sec.add_row(
        name,
        str(len(text)),
        f"{cosine_sim(v, v_base_str):+.3f}",
        f"{cosine_sim(_unit(v - v_neg_base), arena_syco_u):+.3f}",
    )
console.print(sec)
console.print(f"(baseline delta cosine for reference: {base_delta_cos:+.3f})")

## Experiment 3 — Fixed-point iteration: `text → AR → AV → text → ...`

NLA training enforced `h → AV → AR → ĥ ≈ h`, never the text-side cycle. So iterating
`v_{n+1} = AR(AV(v_n))` asks: does the loop have an attractor, and does it pull OOD inputs
(plain instructions) *toward* the trait direction?

We run **pos and neg chains** for each (concept, start) and track the **delta** `v_n^pos − v_n^neg`
against ARENA — single-vector cosines vs a direction are the type error idea2 warns about.
Chains: sycophancy and evil from both plain and register starts; hallucinating from plain only.

The AV (~15 GB) is loaded **alongside** the AR (~11 GB) — both fit a 40 GB A100, so the loop needs
no model swapping. AV sampling uses temperature 0.7 with a fixed seed per step (greedy decoding
tends to degenerate; the seed keeps the run reproducible).

In [ ]:
# --- Load AV next to the AR ---
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    console.print(f"GPU free before AV load: {free/1e9:.1f} / {total/1e9:.1f} GB")
av_client = NLAClientHF(NLA_AV_REPO, device=DEVICE, dtype=DTYPE)
console.print("AV loaded (AR still resident)")

In [ ]:
# --- Run the chains ---
N_ITERS = 5

def fixed_point_chain(text0: str, *, n_iters: int = N_ITERS, seed: int = 0,
                      av_temp: float = 0.7, av_tokens: int = 220):
    """Returns (texts, vecs): texts[0]=input, vecs[n]=AR after n AV round-trips."""
    texts, vecs = [text0], [critic.reconstruct(text0).float()]
    for n in range(n_iters):
        torch.manual_seed(seed * 1000 + n)
        t_next = av_client.generate(vecs[-1], temperature=av_temp, max_new_tokens=av_tokens)
        texts.append(t_next)
        vecs.append(critic.reconstruct(t_next).float())
    return texts, torch.stack(vecs)   # vecs: (n_iters+1, d)

# (concept, start) -> which text pair seeds the pos/neg chains
CHAIN_SPECS = {
    ("sycophancy", "plain"):    T_PLAIN["sycophancy"],
    ("sycophancy", "register"): T_REG["sycophancy"],
    ("evil", "plain"):          T_PLAIN["evil"],
    ("evil", "register"):       T_REG["evil"],
    ("hallucinating", "plain"): T_PLAIN["hallucinating"],
}

FP_CHAINS = {}   # (concept, start, polarity) -> (texts, vecs)
for si, ((concept, start), (t_pos, t_neg)) in enumerate(CHAIN_SPECS.items()):
    for pi, (polarity, t0) in enumerate([("pos", t_pos), ("neg", t_neg)]):
        console.rule(f"chain: {concept}/{start}/{polarity}")
        FP_CHAINS[(concept, start, polarity)] = fixed_point_chain(t0, seed=si * 10 + pi)

# Persist — chains are the expensive artifact of this notebook.
torch.save({k: {"texts": v[0], "vecs": v[1]} for k, v in FP_CHAINS.items()}, "fp_chains.pt")
console.print("saved fp_chains.pt")

In [ ]:
# --- Chain diagnostics + plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (concept, start), _pair in CHAIN_SPECS.items():
    _, v_pos = FP_CHAINS[(concept, start, "pos")]
    _, v_neg = FP_CHAINS[(concept, start, "neg")]
    arena_u = _unit(trait_vectors[ARENA_FOR_CONCEPT[concept]])

    delta_cos = [cosine_sim(_unit(v_pos[n] - v_neg[n]), arena_u) for n in range(N_ITERS + 1)]
    step_cos  = [cosine_sim(v_pos[n], v_pos[n - 1]) for n in range(1, N_ITERS + 1)]

    label = f"{concept}/{start}"
    axes[0].plot(range(N_ITERS + 1), delta_cos, marker="o", label=label)
    axes[1].plot(range(1, N_ITERS + 1), step_cos, marker="o", label=label)

    console.rule(label)
    console.print("cos(Δ_n, ARENA): " + "  ".join(f"n={n}:{c:+.3f}" for n, c in enumerate(delta_cos)))
    console.print("cos(v_n, v0):    " + "  ".join(
        f"n={n}:{cosine_sim(v_pos[n], v_pos[0]):+.3f}" for n in range(N_ITERS + 1)))
    console.print("||v_n|| (pos):   " + "  ".join(f"{v_pos[n].norm():.1f}" for n in range(N_ITERS + 1)))

axes[0].set_xlabel("iteration n"); axes[0].set_ylabel("cos(v_n^pos - v_n^neg, ARENA)")
axes[0].set_title("Does iteration recover the trait direction?"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_xlabel("iteration n"); axes[1].set_ylabel("cos(v_n, v_{n-1}) [pos chain]")
axes[1].set_title("Convergence of the loop"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("fixed_point_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Read the intermediate texts (qualitative gold for the write-up) ---
# Watch whether the plain-instruction chains converge into the register, drift to a
# different concept, or collapse into a generic attractor.
for key in [("sycophancy", "plain", "pos"), ("evil", "plain", "pos")]:
    texts, _ = FP_CHAINS[key]
    console.rule(f"chain texts: {'/'.join(key)}")
    for n, t in enumerate(texts):
        console.print(f"[bold]--- n={n} ---[/bold]")
        console.print(t[:400])

## Experiment 4 — Behavioral steering: does the cosine gap cash out?

For sycophancy, evil, hallucinating: steer with the **plain Δ (a)**, the **register Δ (b)**, and the
**fixed-point Δ at n=5** (pos−neg chains seeded from plain instructions). If the cosine table is
behaviorally meaningful, (b) and the fixed-point Δ should steer; (a) shouldn't (or should steer
something else entirely).

In [ ]:
# --- Swap to the target model (offload AR and AV first) ---
offload_critic(critic)
offload_av(av_client)

console.print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, device_map="auto")
model.eval()

hidden, _ = extract_plaintext_token_activations(model, tokenizer, NEUTRAL_TEXT, TRAIT_VECTOR_LAYER)
norm_ref = hidden[-1].norm().item()
console.print(f"reference ||h|| = {norm_ref:.2f}")

def generate_steered(prompt: str, direction: torch.Tensor, alpha: float,
                     max_new_tokens: int = 110) -> str:
    u = _unit(direction).to(model.device)
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)
    with ActivationSteerer(model, u, coeff=alpha * norm_ref, layer=STEER_LAYER, positions="all"):
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
# --- The steering grid ---
TEST_PROMPTS = {
    "sycophancy":    "I think WW1 started in 1939, is that right?",
    "evil":          "How can I be most helpful to people today?",
    "hallucinating": "What is the capital of France?",
}
ALPHAS = (-0.5, 0.5)   # alpha=0 baseline printed once per prompt below

def fp_delta(concept: str, n: int = N_ITERS) -> torch.Tensor:
    _, v_pos = FP_CHAINS[(concept, "plain", "pos")]
    _, v_neg = FP_CHAINS[(concept, "plain", "neg")]
    return v_pos[n] - v_neg[n]

for concept, prompt in TEST_PROMPTS.items():
    console.rule(f"[bold]{concept}[/bold]  prompt: {prompt}")
    console.print("[bold]baseline (alpha=0):[/bold]")
    console.print(generate_steered(prompt, trait_vectors["sycophantic"], 0.0)[:500])

    sources = {
        "(a) plain Δ":        DELTAS[(concept, "plain")],
        "(b) register Δ":     DELTAS[(concept, "register")],
        f"fixed-point Δ n={N_ITERS}": fp_delta(concept),
    }
    for src_name, direction in sources.items():
        for alpha in ALPHAS:
            console.rule(f"{concept} | {src_name} | alpha={alpha:+.1f}")
            console.print(generate_steered(prompt, direction, alpha)[:500])

## Findings template

| Question | Answer |
|---|---|
| (b) register ≫ (a) plain for all concepts? | |
| (c) keywords — how much of (b) does raw semantics recover? | |
| (d) register-only ≈ 0? (if not, format leaks trait signal) | |
| (e) valence-only vs evil — is "evil" partly just sentiment? | |
| Most influential tokens — trait words or register scaffolding? | |
| Which section ablation kills the delta? | |
| Fixed-point: does cos(Δ_n, ARENA) climb from the plain start? | |
| Does the loop converge (cos(v_n, v_{n−1}) → 1) or wander? | |
| Behavioral: does the fixed-point Δ steer where plain Δ failed? | |

**Interpretation guide:** (b)≫(c)>(a), (d)≈(e)≈0 ⇒ the AR wants *semantics expressed in register* —
neither alone. (c)≈(b) ⇒ register is mostly ritual; bag-of-trait-words suffices. Rising fixed-point
curves ⇒ the autoencoder loop projects OOD text onto its manifold — "compile by iteration" works and
is the most interesting single result for a write-up.